# Mistério em João Pessoa

Um assassinato foi registrado em João Pessoa. A polícia recolheu 6 arquivos crus (`dados/*.csv`) e é com eles que você vai trabalhar — sem nenhum pipeline pronto por trás, dessa vez é você quem constrói.

**Ferramentas: só pandas + MinIO + Jupyter.** Tudo se resolve com pandas puro: `pd.read_csv(...)` pra ler o CSV cru, `df.to_parquet(...)` pra publicar uma tabela no MinIO, `pd.read_parquet(...)` pra ler de volta. Ilustrado com exemplo logo na Fase 1, abaixo. Filtros, joins e agregações são sempre pandas puro (`merge`, `groupby`, filtro booleano...).

## O que você recebeu

6 CSVs em `dados/`:

| Arquivo | O que é |
|---|---|
| `ocorrencia.csv` | O registro da ocorrência (data, tipo, cidade, descrição) |
| `pessoa.csv` | Cadastro de pessoas (nome, endereço) |
| `cnh.csv` | Dados de CNH — quem tem carteira de motorista, placa e veículo |
| `depoimento.csv` | Depoimentos de testemunhas |
| `membro_academia.csv` | Matrículas de uma academia da cidade |
| `checkin_academia.csv` | Check-ins de entrada na academia |

Sim, faltou combinar os formatos de data entre as tabelas — cada uma pode vir de um jeito diferente (texto, ISO, número). É de propósito: parte do trabalho de qualquer engenheiro de dados é notar isso e não deixar passar batido.

## O que você entrega

1. **Fase 1 — Bronze**: os 6 CSVs publicados como tabelas bronze (`df.to_parquet(f"s3://{BUCKET}/bronze/<nome>.parquet", storage_options=STORAGE_OPTIONS)`).
2. **Fase 2 — Investigação**: livre — use pandas (`pd.read_parquet` + `merge`/filtros) para seguir as pistas até chegar a **1** suspeito.
3. **Fase 3 — Resposta final na Silver**: uma tabela `silver.resposta_caso` com sua conclusão (ver Fase 3 no final deste notebook pro formato esperado).

Sem spoiler aqui — a história e as pistas estão só nos dados. Boa investigação!

In [9]:
import os

import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)

# Conexão com o MinIO (S3-compatível) — pandas usa isso direto via s3fs
BUCKET = os.environ.get("LAKEHOUSE_S3_BUCKET", "lakehouse")
STORAGE_OPTIONS = {
    "key": os.environ.get("LAKEHOUSE_S3_ACCESS_KEY", "trilha"),
    "secret": os.environ.get("LAKEHOUSE_S3_SECRET_KEY", "trilha123"),
    "client_kwargs": {"endpoint_url": os.environ.get("LAKEHOUSE_S3_ENDPOINT", "http://minio:9000")},
}

## Fase 1 — Bronze

O padrão pra publicar qualquer CSV cru como tabela bronze é sempre o mesmo — 2 passos, só pandas:

```python
df = pd.read_csv("dados/<arquivo>.csv")                                                                     # lê o CSV cru direto do disco
df.to_parquet(f"s3://{BUCKET}/bronze/<nome_tabela>.parquet", storage_options=STORAGE_OPTIONS, index=False)  # publica: grava Parquet
```

Cada tabela vira **1 arquivo Parquet flat** na camada — `bronze/<nome_tabela>.parquet`, sem subpasta por tabela. Feito isso, dá pra conferir pelo MinIO Console (http://localhost:9001) — os arquivos vão aparecendo em `lakehouse/bronze/`.

Um exemplo pronto (`ocorrencia`), com uma ilustração rápida de como ler de volta logo depois — daí é sua vez de fazer o mesmo padrão para as outras 5 tabelas.

In [10]:
# Exemplo pronto: ocorrencia
df_ocorrencia = pd.read_csv("dados/ocorrencia.csv")
df_ocorrencia.to_parquet(f"s3://{BUCKET}/bronze/ocorrencia.parquet", storage_options=STORAGE_OPTIONS, index=False)
df_ocorrencia

,id,data,tipo,descricao,cidade
0,1,23/02/2026,assassinato,"Corpo encontrado nas imediações da academia Letsvibe Quadramares 24h, bairro de Quadramares, por volta das 09h10. Testemunha 1 mora na casa de numero_endereco mais alto da Avenida Ministro José Américo de Almeida. Testemunha 2 se chama Cida (apelido de Maria Aparecida) e mora na Avenida Rui Carneiro.",João Pessoa
1,2,11/01/2026,furto,Furto de bicicleta relatado na Praça Rio Branco.,João Pessoa
2,3,02/02/2026,vandalismo,Pichação em muro na Avenida Epitácio Pessoa.,João Pessoa
3,4,14/03/2026,furto,Furto em comércio na orla de Tambaú.,João Pessoa
4,5,01/02/2026,assassinato,Ocorrência registrada no centro da cidade vizinha.,Bayeux
5,6,20/02/2026,furto,Furto de veículo relatado no bairro Jardim Oceania.,João Pessoa
6,7,05/03/2026,fraude,Fraude bancária relatada por morador do Bessa.,João Pessoa
7,8,28/01/2026,vandalismo,Depredação de praça pública em Mangabeira.,João Pessoa
8,9,17/02/2026,furto,Furto reportado em Cabedelo.,Cabedelo
9,10,09/03/2026,fraude,Golpe reportado em Santa Rita.,Santa Rita


### Lendo de volta (exemplo rápido)

Pra ler uma tabela publicada é só apontar o `pd.read_parquet` pro mesmo caminho, ilustrado com a tabela que acabamos de publicar:

- `pd.read_parquet(f"s3://{BUCKET}/bronze/<tabela>.parquet", storage_options=STORAGE_OPTIONS)` lê o Parquet inteiro direto do MinIO com pandas — é o que você vai usar na Fase 2 pra ler cada uma das 6 tabelas bronze.

Pra ver o que já existe fisicamente numa camada, sem precisar ler o conteúdo nem escrever código nenhum: abra o **MinIO Console** (http://localhost:9001) e olhe os arquivos em `lakehouse/bronze/`.

In [5]:
# Lendo a tabela de volta direto do MinIO com pandas — deve ser idêntica a df_ocorrencia
pd.read_parquet(f"s3://{BUCKET}/bronze/ocorrencia.parquet", storage_options=STORAGE_OPTIONS)

,id,data,tipo,descricao,cidade
0,1,23/02/2026,assassinato,Corpo encontrado nas imediações da academia Le...,João Pessoa
1,2,11/01/2026,furto,Furto de bicicleta relatado na Praça Rio Branco.,João Pessoa
2,3,02/02/2026,vandalismo,Pichação em muro na Avenida Epitácio Pessoa.,João Pessoa
3,4,14/03/2026,furto,Furto em comércio na orla de Tambaú.,João Pessoa
4,5,01/02/2026,assassinato,Ocorrência registrada no centro da cidade vizi...,Bayeux
5,6,20/02/2026,furto,Furto de veículo relatado no bairro Jardim Oce...,João Pessoa
6,7,05/03/2026,fraude,Fraude bancária relatada por morador do Bessa.,João Pessoa
7,8,28/01/2026,vandalismo,Depredação de praça pública em Mangabeira.,João Pessoa
8,9,17/02/2026,furto,Furto reportado em Cabedelo.,Cabedelo
9,10,09/03/2026,fraude,Golpe reportado em Santa Rita.,Santa Rita


In [12]:
# TODO: repita o padrão da célula do exemplo para "pessoa.csv" -> bronze.pessoa
df_pessoa = pd.read_csv("dados/pessoa.csv")
df_pessoa.to_parquet(f"s3://{BUCKET}/bronze/pessoa.parquet", storage_options=STORAGE_OPTIONS, index=False)
df_pessoa

,id,nome,rua,numero_endereco,bairro
0,1,Genildo Cavalcanti Farias,Rua das Trincheiras,245,Torre
1,2,Rosângela Beltrão Dantas,Avenida João Machado,812,Jaguaribe
2,3,Josenildo Pereira da Rocha,Avenida Ministro José Américo de Almeida,1190,Manaíra
3,4,Maria Aparecida Nunes,Avenida Rui Carneiro,733,Tambaú
4,5,Roberto Carlos Ximenes,Rua Simeão Leal,340,Tambaú
5,6,Genilda Nunes,Avenida Ministro José Américo de Almeida,101,Manaíra
6,7,Patrícia Gonçalves Trindade,Avenida Ministro José Américo de Almeida,551,Manaíra
7,8,Vandeilson Ximenes,Avenida Ministro José Américo de Almeida,259,Manaíra
8,9,Rosilene Araújo Meira,Avenida Ministro José Américo de Almeida,228,Manaíra
9,10,Luzinete Ramos Toscano,Avenida Ministro José Américo de Almeida,115,Manaíra


In [14]:
# TODO: repita o padrão para "cnh.csv" -> bronze.cnh
df_cnh = pd.read_csv("dados/cnh.csv")
df_cnh.to_parquet(f"s3://{BUCKET}/bronze/cnh.parquet", storage_options=STORAGE_OPTIONS, index=False)
df_cnh

,pessoa_id,placa,marca,modelo,cor_veiculo
0,1,KJP4H23,Fiat,Uno,Branco
1,2,MTB2G17,Jeep,Renegade,Cinza
2,4,HLV9N37,Hyundai,HB20,Vermelho
3,6,JJX1A45,Fiat,Uno,Branco
4,8,DWG2V70,Fiat,Mobi,Branco
5,11,LPQ9B02,Fiat,Uno,Vermelho
6,16,JMQ5O76,Hyundai,HB20,Vermelho
7,17,JJW7E64,Fiat,Argo,Branco
8,20,KXD1X72,Fiat,Uno,Branco
9,23,RGT7C61,Fiat,Uno,Prata


In [15]:
# TODO: repita o padrão para "depoimento.csv" -> bronze.depoimento
df_depoimento = pd.read_csv("dados/depoimento.csv")
df_depoimento.to_parquet(f"s3://{BUCKET}/bronze/depoimento.parquet", storage_options=STORAGE_OPTIONS, index=False)
df_depoimento

,id,ocorrencia_id,pessoa_id,depoimento
0,1,1,3,"Eu estava chegando em casa por volta das 9h10 quando vi um homem saindo correndo da Academia Letsvibe Quadramares 24h. Ele usava roupa de treino e tinha uma carteirinha do plano Ouro pendurada na mochila — parecia nova, acho que vi um adesivo de 'bem-vindo' com data de janeiro."
1,2,1,4,"Não vi o rosto dele, mas vi o carro: um Fiat Uno branco, bem velho, saindo em disparada. A placa começava com KJP4, não consegui ver o resto porque estava com lama."
2,3,4,15,"Vi dois homens discutindo perto do comércio, mas não reconheci ninguém."


In [16]:
# TODO: repita o padrão para "membro_academia.csv" -> bronze.membro_academia
df_membro_academia = pd.read_csv("dados/membro_academia.csv")
df_membro_academia.to_parquet(f"s3://{BUCKET}/bronze/membro_academia.parquet", storage_options=STORAGE_OPTIONS, index=False)
df_membro_academia

,matricula_id,pessoa_id,nome_academia,plano,data_matricula
0,LV001,1,Letsvibe Quadramares 24h,ouro,2026-01-08
1,LV002,2,Letsvibe Quadramares 24h,ouro,2026-01-19
2,LV003,5,Letsvibe Quadramares 24h,bronze,2026-01-25
3,LV004,6,Letsvibe Quadramares 24h,ouro,2026-03-02
4,LV005,30,Letsvibe Quadramares 24h,bronze,2025-11-17
5,LV006,12,Letsvibe Quadramares 24h,ouro,2025-12-03
6,LV007,7,Letsvibe Quadramares 24h,bronze,2025-11-22
7,LV008,26,Letsvibe Quadramares 24h,ouro,2026-03-04
8,LV009,22,Letsvibe Quadramares 24h,ouro,2025-12-19
9,LV010,32,Letsvibe Quadramares 24h,bronze,2025-11-20


In [17]:
# TODO: repita o padrão para "checkin_academia.csv" -> bronze.checkin_academia
df_checkin_academia = pd.read_csv("dados/checkin_academia.csv")
df_checkin_academia.to_parquet(f"s3://{BUCKET}/bronze/checkin_academia.parquet", storage_options=STORAGE_OPTIONS, index=False)
df_checkin_academia

,matricula_id,data,hora
0,LV001,23022026,09:05
1,LV002,23022026,09:15
2,LV003,23022026,18:40
3,LV001,18012026,17:56
4,LV001,12012026,13:23
...,...,...,...
73,LV020,8032026,19:53
74,LV020,1032026,20:57
75,LV020,18012026,21:28
76,LV020,9022026,13:53


Checagem: as 6 tabelas devem aparecer em `lakehouse/bronze/` no **MinIO Console** (http://localhost:9001).

## Fase 2 — Investigação (pandas)

A partir de aqui é livre: leia as tabelas bronze direto do MinIO com `pd.read_parquet(f"s3://{BUCKET}/bronze/<tabela>.parquet", storage_options=STORAGE_OPTIONS)`, e siga as pistas com `merge`/filtros de DataFrame, do jeito que preferir.

Um roteiro sugerido (não obrigatório seguir exatamente esta ordem, mas ajuda a não se perder):

1. Ache a ocorrência (tipo = assassinato, cidade = João Pessoa) e leia a `descricao` com atenção — ela dá pistas de **endereço** para achar testemunhas em `pessoa`.
2. Ache as testemunhas em `pessoa` e cruze com `depoimento` para ler o que cada uma contou.
3. Cada depoimento traz uma pista diferente — uma aponta para `membro_academia` (algo sobre o plano/data de matrícula), a outra para `cnh` (algo sobre o veículo/placa).
4. Filtre cada tabela pela pista correspondente. Sozinha, cada pista pode sobrar **mais de 1** candidato — o cruzamento das duas é que deve fechar em exatamente **1** pessoa.
5. (Bônus) Confirme em `checkin_academia` que o suspeito realmente esteve na academia por perto do horário da ocorrência.

Lembre-se: as datas não vêm todas no mesmo formato entre as tabelas — repare em cada uma antes de comparar/filtrar por data.

In [19]:
# Lendo todas as tabelas bronze direto do MinIO via pandas
ocorrencia = pd.read_parquet(f"s3://{BUCKET}/bronze/ocorrencia.parquet", storage_options=STORAGE_OPTIONS)
# pessoa = pd.read_parquet(f"s3://{BUCKET}/bronze/pessoa.parquet", storage_options=STORAGE_OPTIONS)
# cnh = pd.read_parquet(f"s3://{BUCKET}/bronze/cnh.parquet", storage_options=STORAGE_OPTIONS)
# depoimento = pd.read_parquet(f"s3://{BUCKET}/bronze/depoimento.parquet", storage_options=STORAGE_OPTIONS)
# membro_academia = pd.read_parquet(f"s3://{BUCKET}/bronze/membro_academia.parquet", storage_options=STORAGE_OPTIONS)
# checkin_academia = pd.read_parquet(f"s3://{BUCKET}/bronze/checkin_academia.parquet", storage_options=STORAGE_OPTIONS)

ocorrencia

,id,data,tipo,descricao,cidade
0,1,23/02/2026,assassinato,"Corpo encontrado nas imediações da academia Letsvibe Quadramares 24h, bairro de Quadramares, por volta das 09h10. Testemunha 1 mora na casa de numero_endereco mais alto da Avenida Ministro José Américo de Almeida. Testemunha 2 se chama Cida (apelido de Maria Aparecida) e mora na Avenida Rui Carneiro.",João Pessoa
1,2,11/01/2026,furto,Furto de bicicleta relatado na Praça Rio Branco.,João Pessoa
2,3,02/02/2026,vandalismo,Pichação em muro na Avenida Epitácio Pessoa.,João Pessoa
3,4,14/03/2026,furto,Furto em comércio na orla de Tambaú.,João Pessoa
4,5,01/02/2026,assassinato,Ocorrência registrada no centro da cidade vizinha.,Bayeux
5,6,20/02/2026,furto,Furto de veículo relatado no bairro Jardim Oceania.,João Pessoa
6,7,05/03/2026,fraude,Fraude bancária relatada por morador do Bessa.,João Pessoa
7,8,28/01/2026,vandalismo,Depredação de praça pública em Mangabeira.,João Pessoa
8,9,17/02/2026,furto,Furto reportado em Cabedelo.,Cabedelo
9,10,09/03/2026,fraude,Golpe reportado em Santa Rita.,Santa Rita


### Passo 1 — a ocorrência

Filtre `ocorrencia` para achar o assassinato em João Pessoa, e leia a `descricao` inteira (`print(...)` ajuda a não truncar o texto).

In [ ]:
# TODO: filtre ocorrencia por tipo == "assassinato" e cidade == "João Pessoa"
# e dê print() na coluna "descricao" da linha encontrada.


### Passo 2 — as testemunhas

A descrição da ocorrência aponta para 2 pessoas em `pessoa`, cada uma identificada por uma pista de **rua/endereço** diferente (uma pelo número mais alto numa rua, a outra pelo nome numa outra rua). Ache as duas, depois cruze os `id` delas com `depoimento.pessoa_id` para ler o que cada uma contou.

In [ ]:
# TODO: ache as 2 testemunhas em `pessoa` (via as pistas de rua/endereço da descrição)
# e depois os depoimentos delas em `depoimento` (merge por pessoa_id, ou filtro direto).


### Passo 3 — duas pistas, duas tabelas

Um depoimento descreve algo sobre o **plano e a data de matrícula** de alguém na academia — filtre `membro_academia` por isso. Sozinha, essa pista deve deixar **mais de 1** candidato (tudo bem, é assim mesmo).

O outro depoimento descreve algo sobre o **veículo/placa** de quem fugiu — filtre `cnh` por isso.

In [ ]:
# TODO: filtre `membro_academia` pela pista do 1o depoimento (plano + data de matrícula)
# candidatos_academia = ...


In [ ]:
# TODO: filtre `cnh` pela pista do 2o depoimento (veículo/placa)
# candidatos_placa = ...


### Passo 4 — cruzando as pistas

Cruze `candidatos_academia` com `candidatos_placa` (por `pessoa_id`/`id`) — deve sobrar exatamente **1** pessoa. Se sobrar mais de 1 (ou 0), revise os filtros dos passos anteriores.

In [ ]:
# TODO: cruze os dois conjuntos de candidatos e confira que sobrou 1 só suspeito
# suspeito = ...


### Passo 5 (bônus) — confirmar com o check-in

Cruze `membro_academia` com `checkin_academia` (por `matricula_id`) e confira que o suspeito tem um check-in na data da ocorrência, num horário compatível com o depoimento. Repare no formato da data em `checkin_academia` — é diferente do formato usado em `ocorrencia`.

In [ ]:
# TODO (bônus): confirme o check-in do suspeito na data/horário da ocorrência


## Fase 3 — Resposta final na Silver

Chegou a hora de publicar sua conclusão como uma tabela — o entregável desta tarefa. Monte um DataFrame de **1 linha** com estas colunas:

| coluna | conteúdo |
|---|---|
| `nome_suspeito` | o nome completo da pessoa em `pessoa` |
| `placa_veiculo` | a placa (de `cnh`) que fechou o caso |
| `pista_academia` | qual plano/data de matrícula bateu com o depoimento |
| `pista_veiculo` | qual detalhe do veículo bateu com o depoimento |
| `justificativa` | 1-2 frases explicando o raciocínio (pode ser texto livre) |

E publique com `df_resposta.to_parquet(f"s3://{BUCKET}/silver/resposta_caso.parquet", storage_options=STORAGE_OPTIONS, index=False)` — depois disso, o **MinIO Console** (http://localhost:9001), em `lakehouse/silver/resposta_caso.parquet`, já mostra o resultado.

In [ ]:
# TODO: monte df_resposta (1 linha, colunas da tabela acima) e publique na silver
# df_resposta = pd.DataFrame({...})
# df_resposta.to_parquet(f"s3://{BUCKET}/silver/resposta_caso.parquet", storage_options=STORAGE_OPTIONS, index=False)


---

Terminou? `lakehouse/bronze/` deve ter as 6 tabelas desta tarefa, e `lakehouse/silver/resposta_caso.parquet` deve ter sua conclusão — dá pra confirmar tudo pelo MinIO Console (http://localhost:9001) sem precisar de mais nada.